In [1]:
#Parameters

Scenario = None
T_planet = None
R_planet = None
Z_planet = None
G_planet = None
Age = None
Sep_p = None

Mag_planet_K= None

Mag_star_K = None

folder_name = None

day = None

In [2]:
#import the libraries 

import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import fftpack,signal
import matplotlib
from matplotlib.ticker import ScalarFormatter
from astropy.visualization import (MinMaxInterval, SqrtStretch, ImageNormalize, ZScaleInterval)
import scipy
import json
import subprocess
import os
import tqdm

from exocrires import spectra_2d
from exocrires import info
from exocrires import analysis


path_a='/Users/richard/project_EXOCRIRES'
path='/Users/richard/project_EXOCRIRES/%s'%folder_name
lambda_range=info.lambda_range

In [ ]:
#with the new version of ETC, it is possible to input the magnitude and generate outputs in a different band
#input wavelength setting
lambda_range=info.lambda_range
input_json_name='/H1800_10_1.51_6.60.json'
data_pp=spectra_2d.load_json(path_a+input_json_name,'r')


#stellar_mag=[['Y', np.round(m_s_y,2)],['J',7.31],['H', 6.91],['K', 6.41], ['L', 999]]
stellar_mag=[['Y', 999], ['J',999],['H',999],['K', Mag_star_K], ['L', 999]]
name_model='%s-%s-%s'%(T_planet, G_planet, Z_planet)



In [ ]:
# The sky background distribution should be the same for each run of output data. Therefore we only need to run it once with any input specgtrum of the planet
##-----Combine with the sky background, the stellar light and the noise terms-----##
input_json_name='H1800_10_1.51_6.60.json'
key=lambda_range['key'][-3]


filename=path+'/spectra_input/upload_lowres_star_%s_10km_rot_lsf.dat'%key

dit= data_pp['timesnr']['DET1.DIT']
ndit= data_pp['timesnr']['DET1.NDIT']
nspec=1
nspat=105 #nspat for aperture 105
dark=0.003 #e-/pix/s, width of the distribution
read=6 #e-/pix, width of the distribution
npix=nspec*nspat
ndark=dit*dark #e-/pix/exposure
po=Sep_p #pixel offset of the planet position on the detector, 1 pixel = 0.059 arcsec

col_names=[]

for col in range(18):

    col_names.append('dat_diff_%s'%col)

    print (col_names[col])
    
for x in range(3,4):
    
    key=lambda_range['key'][x]
    
    print ('The combination is running on %s'%key)

    data_log=pd.read_csv(path+'/%s/star/%s/stellar_simu_%s.csv'%(day,key,key))
    data_log=data_log.replace(0, 1e-10)
    data_log=data_log.fillna(1e-10)
    data_log[data_log<0]=1e-10
    #--Use the next three lines when generating planet-only noise-free 2-D spectra for subtraction later on-- 
    #data_log=np.zeros(shape=(6138*7,18))
    #data_log=pd.DataFrame(data_log,columns=col_names)
    #data_log.insert(0, 'wavelength(nm)', data_log_p['wavelength(nm)'])

    data_log_p=pd.read_csv(path+'/%s/planet/%s/%s/planet_simu.csv'%(day,name_model, key))
    data_log_p=data_log_p.replace(0, 1e-10)
    data_log_p=data_log_p.fillna(1e-10)
    data_log_p[data_log_p<0]=1e-10
    #--Use the next three lines when generating star-only noise-free 2-D spectra for subtraction later on-- 
    #data_log_p=np.zeros(shape=(6138*7,18))
    #data_log_p=pd.DataFrame(data_log_p,columns=col_names) 
    #data_log_p.insert(0, 'wavelength(nm)', data_log['wavelength(nm)'])



    
    data_log_k=pd.read_csv(path+'/%s/sky/%s/sky_simu.csv'%(day,key))
    data_log_k=data_log_k.replace(0,1e-10)
    data_log_k=data_log_k.fillna(1e-10)
    data_log_k[data_log_k<0]=1e-10
    #--Use the next three lines when generating planet-only noise-free 2-D spectra for subtraction later on-- 
    #data_log_k=np.zeros(shape=(6138*7,18))
    #data_log_k=pd.DataFrame(data_log_k,columns=col_names) 
    #data_log_k.insert(0, 'wavelength(nm)', data_log_p['wavelength(nm)'])
    
    
    simu_planet=spectra_2d.spectra_2d (path_etc_local=path+'/etc_cli.py', path_etc_input=path+input_json_name, path_input_modi=path+'/modi_input_%s_p_AO.json'%day,\
             path_etc_output=path+'/%s/planet/%s/%s'%(day, name_model, key), target='planet', aperture_size=35, date='%s'%day,\
        order_window=lambda_range['wave'][x],\
        order_number=[m for m in lambda_range['order'][x]],\
        model_name=filename)
    

    nodding={'A':1024-int(5/0.059), 'B':1024+int(5/0.059)}

    

    for z in nodding:

        if ndit <=100:
            

            tot_data_com=simu_planet.combine(data_star=data_log, data_sky=data_log_k, data_p=data_log_p,\
                                                    noise=True, dark=ndark, ron=read, d=po, star_posi=nodding[z], \
                        plane=(2048,2048*3), plot_combination=False, ceil_percentage=99, NDIT=ndit)

            np.save(path+'/%s/planet/%s/%s/combine_focal_plane_%s.npy'%(day,name_model,key, z), tot_data_com)


        elif ndit > 100:

            n=40

            print ('Too many exposures. We stack every %s exposures and merge all into one npy file .later'%n)

            ndit_divide = ndit//n

            ndit_remain = ndit % n
        
            #tot_data_com=np.zeros(shape=(len(lambda_range['wave'][x]), ndit, 2048, 2048*3))
            
            for i in tqdm.tqdm(range(ndit_divide)):

                tot_data_com_serie=simu_planet.combine(data_star=data_log, data_sky=data_log_k, data_p=data_log_p,\
                                                                noise=True, dark=ndark, ron=read, d=po, star_posi=nodding[z], \
                                        plane=(2048,2048*3), plot_combination=False, NDIT=n)
                
                #tot_data_com[:,i*n:(i+1)*n,:,:]=tot_data_com_serie

                np.save(path+'/%s/planet/%s/%s/combine_focal_plane_%s_%s.npy'%(day,name_model,key, z, i), tot_data_com_serie)

                #fig.savefig(path+'/%s/planet/%s/%s/combine_%s.png'%(day, name_model, key, z, n))
            
            if ndit_remain > 0:
                
                tot_data_com_seire=simu_planet.combine(data_star=data_log, data_sky=data_log_k, data_p=data_log_p,\
                                    noise=True, dark=ndark, ron=read, d=po, star_posi=nodding[z], \
                                    plane=(2048,2048*3), plot_combination=False, ceil_percentage=99, NDIT=ndit_remain)

                np.save(path+'/%s/planet/%s/%s/combine_focal_plane_%s_%s.npy'%(day,name_model,key, z, ndit_remain), tot_data_com_serie)

                #fig.savefig(path+'/%s/planet/%s/%s/combine_%s.png'%(day, name_model, key, z, n))

            #tot_data_com[-n_remain:]=tot_data_com_serie



        #fig.savefig(path+'/%s/planet/%s/%s/combine_%s.png'%(day, name_model, key, z))

        #np.save(path+'/%s/planet/%s/%s/combine_focal_plane_%s.npy'%(day,name_model,key, z), tot_data_com)

    print ('Combined 2d spectrum done:%s'%simu_planet.path_output)

    ''''
    #------save the PSF from Noise-free images---------

    noise_free_data=np.load(path+'/%s/planet/%s/%s/noise_free/combine_stack_B.npy'%(day,filename[-33:-23],key))

    for i in range(noise_free_data.shape[0]):

        for j in range(3):

            detector=noise_free_data[i,:,j*2048:(j+1)*2048]

            PSF=detector[1108-17:1108+18, 1000]

            PSF_norm = PSF/np.trapz(PSF)

            np.save(path+'/%s/planet/%s/%s/noise_free/PSF_%s_%s.npy'%(day,filename[-33:-23],key, 29-i, j+1), PSF_norm)
    '''

#clean cache and empty the memory
import gc
gc.collect()
matplotlib.pyplot.close('all')
gc.enable()
gc.collect()


In [ ]:
#Branch_1

key=lambda_range['key'][-3]
nodding={'A':1024-int(5/0.059), 'B':1024+int(5/0.059)}

#Load Position A series adn Position B series

ndit= data_pp['timesnr']['DET1.NDIT']

print ('We need to generate in total %s exposures and sum stack all together in to a single frame.'%ndit)

if ndit <= 100: 

    position_A=np.load(path+'/%s/planet/%s/%s/combine_focal_plane_A.npy'%(day,name_model,key))
    position_B=np.load(path+'/%s/planet/%s/%s/combine_focal_plane_B.npy'%(day,name_model,key))
    N=position_A.shape[1]

    ABBA_series=np.zeros(shape=(position_A.shape[0], N*2, 35, position_A.shape[3]))

    for n in tqdm.tqdm(range(N)):

        A = position_A[:,n]

        B = position_B[:,n]

        #subtract
        C=A-B
        D=B-A

        #double the signal
        central_A=nodding['A']
        central_B=nodding['B']

        C_double=C[:, central_A-17:central_A+18, :]-C[:, central_B-17:central_B+18, :]

        D_double=D[:, central_B-17:central_B+18, :]-D[:, central_A-17:central_A+18, :]

        ABBA_series[:, n*2]=C_double
        ABBA_series[:, n*2+1]=D_double

    np.save(path+'/%s/planet/%s/%s/ABBA_series.npy'%(day,name_model,key), ABBA_series)

    ABBA_stack=np.sum(ABBA_series, axis=1)
    np.save(path+'/%s/planet/%s/%s/ABBA_stack.npy'%(day,name_model,key), ABBA_stack)

else:
    
    chunksize=40


    print ('Too many exposures. We separate for %s each.'%chunksize )

    

    ndit_divide = ndit//chunksize

    ndit_remain = ndit % chunksize

    if ndit_remain ==0:
        ABBA_stack_series=np.zeros(shape=(7, ndit_divide, 35, 2048*3))
    else: 
        ABBA_stack_series=np.zeros(shape=(7, 1+ndit_divide, 35, 2048*3))

    for i in tqdm.tqdm(range(ndit_divide)):

        position_A=np.load(path+'/%s/planet/%s/%s/combine_focal_plane_A_%s.npy'%(day,name_model,key,i))

        position_B=np.load(path+'/%s/planet/%s/%s/combine_focal_plane_B_%s.npy'%(day,name_model,key,i))

        ABBA_series=np.zeros(shape=(position_A.shape[0], chunksize*2, 35, position_A.shape[3]))

        for n in tqdm.tqdm(range(position_A.shape[1])):

            A = position_A[:,n]

            B = position_B[:,n]

            #subtract
            C=A-B
            D=B-A

            #double the signal
            central_A=nodding['A']
            central_B=nodding['B']

            C_double=C[:, central_A-17:central_A+18, :]-C[:, central_B-17:central_B+18, :]

            D_double=D[:, central_B-17:central_B+18, :]-D[:, central_A-17:central_A+18, :]

            ABBA_series[:, n*2]=C_double
            ABBA_series[:, n*2+1]=D_double

        mask_inf=np.isinf(ABBA_series)
        ABBA_series[mask_inf]=np.nan

        ABBA_stack_series[:,i,:,:]=np.sum(ABBA_series, axis=1)



        del (position_A, position_B, ABBA_series)




    ABBA_stack_all=np.sum(ABBA_stack_series, axis=1)


    np.save(path+'/%s/planet/%s/%s/ABBA_stack.npy'%(day,name_model,key), ABBA_stack_all)
    np.save(path+'/%s/planet/%s/%s/ABBA_stack_series.npy'%(day,name_model,key), ABBA_stack_series)     


In [ ]:
font = {'size': 15}

plt.rcParams.update({'font.size': font['size']})

plt.imshow(ABBA_series[1][-2], aspect='auto')
plt.show()

plt.plot(ABBA_series[1][-2][0:35,1000])
plt.show()

plt.imshow(position_B[1][-2][1024+84-20:1024+84+20,:],  aspect='auto')
plt.show()

plt.plot(position_B[1][-2][1024+84+5, :]*2-ABBA_series[-3][-2][17+5,:])
plt.show()

#Delete_old = input("Do you want to delete the files containing two nodding positions? (yes/no): ").strip().lower()
Delete_old = 'yes'  # Set to 'yes' to delete old files
if Delete_old == 'yes':
    if ndit <= 100:
        os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_A.npy'%(day,name_model,key))
        os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_B.npy'%(day,name_model,key))
    else:
        ndit_divide = ndit//40
        ndit_remain = ndit % 40
        for i in tqdm.tqdm(range(ndit_divide)):
            os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_A_%s.npy'%(day,name_model,key,i))
            os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_B_%s.npy'%(day,name_model,key,i))

        if ndit_remain > 0:
            os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_A_%s.npy'%(day,name_model,key,ndit_remain))
            os.remove(path+'/%s/planet/%s/%s/combine_focal_plane_B_%s.npy'%(day,name_model,key,ndit_remain))

In [ ]:
gc.collect()
matplotlib.pyplot.close('all')
gc.enable()
gc.collect()